# The driven damped pendulum (Taylor Chapter 12)

*Class notebook for Chapter 12, and the program for HW13 Problems 12.6, 12.7 (and the optional 12.8-12.10).* Run each cell in order with **Shift-Enter**. You only need to edit the cells marked **Your turn**.

A pendulum with damping, pushed by a sinusoidal drive, with $\sin\phi$ left as $\sin\phi$, obeys Taylor's (12.11):
$$\ddot\phi + 2\beta\dot\phi + \omega_0^2\sin\phi = \gamma\,\omega_0^2\cos(\omega t).$$
The drive strength $\gamma = F_0/(mg)$ compares the drive with the weight. Throughout Chapter 12 Taylor uses
$$\omega = 2\pi \text{ rad/s (so the drive period is 1 s)}, \qquad \omega_0 = 1.5\,\omega, \qquad \beta = \omega_0/4 ,$$
and so does this notebook. Every time axis is in seconds, which here is the same as counting drive periods.

There is no formula for $\phi(t)$, so we solve (12.11) numerically and look at the answers.

## 1. Predict first (before you run anything)

Double-click the cell below this one, type your answers, then Shift-Enter. A sentence each is plenty.

- With a drive weaker than the weight ($\gamma < 1$), start the pendulum from three different angles. After a few drive periods, are the three motions different or the same?
- Once the start-up wiggles have died away, does the motion always repeat with the drive's period, 1 s? If not, what else could it do?
- Suppose you look at the pendulum only once per drive period, at $t = 0, 1, 2, \ldots$ s, and mark a dot each time. How many different dots do you see if the motion repeats every 2 drive periods? If it never repeats?

*(your prediction here)*

## 2. Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact

omega = 2 * np.pi        # rad/s, drive frequency (drive period 1 s)
omega0 = 1.5 * omega     # rad/s, natural frequency
beta = omega0 / 4        # 1/s, damping constant
samples_per_second = 200

## 3. The solver

`solve_ddp` ("driven damped pendulum") is the one function everything else uses. You give it a drive strength `gamma`, a starting angle `phi0` (radians), and how long to run `t_max` (seconds); it starts the pendulum at rest unless you also give `phidot0`.

In [ ]:
def solve_ddp(gamma, phi0, t_max, phidot0=0.0):
    """Solve (12.11) from phi(0) = phi0, phidot(0) = phidot0.
    Returns times t (s), angles phi (rad) and angular velocities phidot (rad/s)."""

    def rates(t, state):
        phi, phidot = state
        phiddot = (-2 * beta * phidot - omega0**2 * np.sin(phi)
                   + gamma * omega0**2 * np.cos(omega * t))
        return [phidot, phiddot]

    t = np.linspace(0, t_max, int(samples_per_second * t_max) + 1)
    sol = solve_ivp(rates, (0, t_max), [phi0, phidot0], t_eval=t,
                    rtol=1e-10, atol=1e-10)
    return sol.t, sol.y[0], sol.y[1]

Line by line:

- The solver wants a second-order equation as two first-order ones, so the "state" is the pair (angle, angular velocity), just as in the HW01 skateboard notebook. `rates` says how fast each one changes: the angle changes at `phidot`, and `phidot` changes at the $\ddot\phi$ that (12.11) gives.
- `np.linspace(...)` asks for the answer 200 times per second, so that every whole second ($t = 0, 1, 2, \ldots$, once per drive period) is one of the sample times. Section 6 uses that.
- `rtol` and `atol` set how accurate the solver must be. They are tight on purpose: in Chapter 12 small numerical errors can grow, so we pay for accuracy.
- The function hands back three arrays; `t, phi, phidot = solve_ddp(...)` catches all three.

## 4. A first run

A weak drive, $\gamma = 0.2$, from rest at $\phi = 0$ (Taylor's Figure 12.2):

In [ ]:
t, phi, phidot = solve_ddp(0.2, 0.0, 6)

plt.figure(figsize=(8, 3.5))
plt.plot(t, phi)
plt.xlabel("t (seconds)")
plt.ylabel("phi (radians)")
plt.title("gamma = 0.2, phi(0) = 0")
plt.grid(alpha=0.3)
plt.show()

After about one drive period the start-up wiggle is gone, and what is left repeats every 1 s, like the linear driven oscillator of Chapter 5.

## 5. Same place, different number: wrapping $\phi$

The angles $\phi$ and $\phi + 2\pi$ are the same position of the pendulum. The solver doesn't know that: if the pendulum goes over the top, $\phi$ just keeps counting. Here is a strong drive, $\gamma = 1.4$, which sends the pendulum over the top once every drive period (Taylor's Figure 12.19). Its $\phi$ climbs (or falls) without end:

In [ ]:
def wrap(phi):
    """The same angle, moved into -pi < phi <= pi by adding or subtracting whole turns (2 pi)."""
    return np.pi - np.mod(np.pi - phi, 2 * np.pi)


print(wrap(7.0), "which is 7 - 2 pi =", 7.0 - 2 * np.pi)
print(wrap(-np.pi), "and", wrap(np.pi), "(both pi: -pi and pi are the same place)")

`np.mod(x, 2 * np.pi)` is the remainder after removing whole multiples of $2\pi$, a number between 0 and $2\pi$; the two `np.pi -` around it shift that range to $(-\pi, \pi]$.

`plot_runs` puts both pictures side by side for one or more starting angles: on the left $\phi$ as the solver computed it, on the right the same runs wrapped. The wrapped curves are drawn as dots, because a line would draw a vertical stroke each time the angle jumps from $\pi$ to $-\pi$. (So a swing whose crest reaches just past $\pi$ shows a few dots along the bottom edge: the same place, under its other label.)

In [ ]:
def plot_runs(gamma, start_angles, t_max):
    """phi(t) for each starting angle (released from rest): as computed, and wrapped."""
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    for phi0 in start_angles:
        t, phi, phidot = solve_ddp(gamma, phi0, t_max)
        left.plot(t, phi, label=f"phi(0) = {phi0:.3f}")
        right.plot(t, wrap(phi), ".", markersize=1.5)
    left.set_title(f"gamma = {gamma}: phi as computed")
    right.set_title("the same runs, wrapped into (-pi, pi]")
    right.set_ylim(-np.pi, np.pi)
    for ax in (left, right):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    left.legend()
    plt.show()


plot_runs(1.4, [0.0], 8)

On the left, $\phi$ drops by about $2\pi$ every second: the pendulum goes all the way round once per drive period. On the right the same motion is a pattern that repeats every second.

## 6. Looking once per drive period: the period of the settled motion

To find out how often the settled motion repeats, look at the pendulum once per drive period ("strobe" it): at $t = 0, 1, 2, \ldots$ s, skipping the first `t_settle` seconds while the start-up behavior dies away. If the strobed angle comes back to the same value every $n$ looks (allowing for whole turns), the motion has period $n$ drive periods.

- `strobe` picks every 200th sample (one per second) after `t_settle` and wraps the angles.
- `find_period` tries $n = 1, 2, 3, \ldots$ and returns the first $n$ for which every strobed angle matches the one $n$ looks later, to within `tolerance` radians. It returns `None` if nothing up to `longest` fits: either the motion never repeats, or it has not settled yet.

In [ ]:
def strobe(t, phi, phidot, t_settle):
    """phi (wrapped) and phidot once per drive period, for t >= t_settle."""
    t_once = t[::samples_per_second]
    keep = t_once >= t_settle
    phi_once = phi[::samples_per_second][keep]
    phidot_once = phidot[::samples_per_second][keep]
    return wrap(phi_once), phidot_once


def find_period(phi_strobe, tolerance=1e-3, longest=16):
    """Smallest n with phi(t + n) = phi(t) for every strobe (up to whole turns), or None."""
    for n in range(1, longest + 1):
        shift = wrap(phi_strobe[n:] - phi_strobe[:-n])
        if np.all(np.abs(shift) < tolerance):
            return n
    return None

Pick a drive strength and a starting angle. The left panel is the first 20 s; the right panel is the last 10 s of a 364 s run, with the strobe times marked as dots. Each choice takes a second or two to compute.

These are the drive strengths from class (Taylor Figures 12.5-12.8): try them from both starting angles. Two things to watch for: the period doubling as $\gamma$ creeps up from 1.078 toward 1.083, and $\gamma = 1.077$, where the starting angle decides which settled motion you get.

In [ ]:
gamma_choices = [1.077, 1.078, 1.081, 1.082, 1.0826, 1.105]
start_choices = {"phi(0) = -pi/2": -np.pi / 2, "phi(0) = 0": 0.0}
t_settle = 300


@interact(gamma=gamma_choices, phi0=start_choices)
def explore(gamma=1.078, phi0=-np.pi / 2):
    t, phi, phidot = solve_ddp(gamma, phi0, t_settle + 64)
    phi_s, phidot_s = strobe(t, phi, phidot, t_settle)
    n = find_period(phi_s)

    fig, (early, late) = plt.subplots(1, 2, figsize=(11, 3.8))
    first = t <= 20
    early.plot(t[first], phi[first])
    early.set_title(f"gamma = {gamma}: the first 20 s")
    last = t >= t[-1] - 10
    late.plot(t[last], wrap(phi[last]), ".", markersize=1.5)
    late.plot(np.arange(t[-1] - 10, t[-1] + 0.5), phi_s[-11:], "ro", label="strobe")
    late.set_title("the last 10 s, wrapped")
    late.legend(loc="upper right")
    for ax in (early, late):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    plt.show()
    print("last 8 strobed angles (rad):", np.round(phi_s[-8:], 4))
    print("period (drive periods):", n)

## 7. The Poincare section

Now plot each strobed state as a dot in the $(\phi, \dot\phi)$ plane: one dot per drive period, hundreds of drive periods (a Poincare section, Taylor 12.8). A motion with period $n$ keeps landing on the same $n$ dots. A motion that never repeats gives dots that never land on each other, but they do not fill the plane either. With 600 dots this takes a few seconds.

In [ ]:
@interact(gamma=gamma_choices)
def poincare(gamma=1.082):
    t, phi, phidot = solve_ddp(gamma, -np.pi / 2, t_settle + 600)
    phi_s, phidot_s = strobe(t, phi, phidot, t_settle)

    plt.figure(figsize=(6, 5))
    plt.plot(phi_s, phidot_s, ".", markersize=3)
    plt.xlabel("phi (radians, wrapped)")
    plt.ylabel("phidot (rad/s)")
    plt.title(f"gamma = {gamma}, phi(0) = -pi/2: one dot per drive period")
    plt.grid(alpha=0.3)
    plt.show()

## 8. Your turn: Problems 12.6 and 12.7

`plot_runs` (section 5) does everything these two problems need. Set the drive strength, the list of starting angles (radians, all released from rest), and the run time from the problem; then run the cell. For 12.7, change the three lines and run it again (or copy the cell).

In [ ]:
# Change the line below.
gamma = 0.2
# Change the line below.
start_angles = [0.0]
# Change the line below.
t_max = 6.0

plot_runs(gamma, start_angles, t_max)

## 9. Your turn (optional): Problems 12.8-12.10

`plot_early_late` solves for $0 < t < 50$ s and plots two windows: the first `t_early` seconds, and $40 < t < 50$ s, when the start-up behavior is long gone. It also prints the wrapped strobe angles at $t = 40, 41, \ldots, 50$ s, so you can read the period off the numbers as well as the picture (or call `find_period` on them, section 6).

In [ ]:
def plot_early_late(gamma, phi0, t_early):
    t, phi, phidot = solve_ddp(gamma, phi0, 50)
    fig, (early, late) = plt.subplots(1, 2, figsize=(11, 3.5))
    first = t <= t_early
    early.plot(t[first], phi[first])
    early.set_title(f"gamma = {gamma}, phi(0) = {phi0:.3f}: 0 < t < {t_early} s")
    last = t >= 40
    late.plot(t[last], phi[last])
    late.set_title("40 < t < 50 s")
    for ax in (early, late):
        ax.set_xlabel("t (seconds)")
        ax.set_ylabel("phi (radians)")
        ax.grid(alpha=0.3)
    plt.show()
    phi_s, phidot_s = strobe(t, phi, phidot, 40)
    print("wrapped phi at t = 40, 41, ..., 50 s:", np.round(phi_s, 4))

In [ ]:
# Change the line below.
gamma = 0.2
# Change the line below.
phi0 = 0.0
# Change the line below.
t_early = 10

plot_early_late(gamma, phi0, t_early)

For 12.10, `plot_runs` with `gamma = 1.073`, `t_max = 50` and a list of starting angles between $-\pi$ and $\pi$ puts many starts on one picture; `strobe` and `find_period` (section 6) give the numbers.

## 10. What you found

Double-click each answer cell, type, Shift-Enter. Put the plots (screenshots are fine) and these answers in your HW13 PDF.

**a. 12.6.** Do your three runs bear out the claim that all solutions approach the same periodic attractor? After about how many drive periods? Compare with your first prediction.

*(your answer here)*

**b. 12.7.** Is there still a unique attractor at $\gamma = 1.06$? Use both panels of your plot in the answer.

*(your answer here)*

**c. 12.8-12.10 (optional).** What is the period of the long-term motion, and does it depend on where you start?

*(your answer here)*

**d. What didn't work** the first time, if anything.

*(your answer here)*